# 0. Problem
## 1211. Queries Quality and Percentage — Easy
Per query name, calculate quality = average(rating / position) and poor query percentage = percentage of rows with rating < 3. Round both to 2 decimals.
Official: https://leetcode.com/problems/queries-quality-and-percentage/

# 1. Setup

In [ ]:
import pandas as pd
queries_rows=[("Dog","Golden Retriever",1,5),("Dog","German Shepherd",2,5),("Dog","Mule",200,1),("Cat","Shirazi",5,2),("Cat","Siamese",3,3),("Cat","Sphynx",7,4)]
queries_pd=pd.DataFrame(queries_rows,columns=["query_name","result","position","rating"]); queries_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate(); queries_spark=spark.createDataFrame(queries_rows,["query_name","result","position","rating"]); queries_spark.createOrReplaceTempView("Queries")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""SELECT query_name,ROUND(AVG(rating*1.0/position),2) AS quality,ROUND(AVG(CASE WHEN rating<3 THEN 1.0 ELSE 0.0 END)*100,2) AS poor_query_percentage FROM Queries GROUP BY query_name ORDER BY query_name"""); sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
work_pd=queries_pd.assign(ratio=queries_pd["rating"]/queries_pd["position"],poor=queries_pd["rating"].lt(3).astype(float)); result_pd=work_pd.groupby("query_name",as_index=False).agg(quality=("ratio","mean"),poor_query_percentage=("poor","mean")); result_pd["quality"]=result_pd["quality"].round(2); result_pd["poor_query_percentage"]=(result_pd["poor_query_percentage"]*100).round(2); result_pd=result_pd.sort_values("query_name").reset_index(drop=True); result_pd

# 4. PySpark Solution

In [ ]:
result_spark=(queries_spark.groupBy("query_name").agg(F.round(F.avg(F.col("rating")/F.col("position")),2).alias("quality"),F.round(F.avg(F.when(F.col("rating")<3,1.0).otherwise(0.0))*100,2).alias("poor_query_percentage")).orderBy("query_name")); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| ratio average | `AVG(rating/position)` | derived column + mean | `avg(col/col)` |
| conditional percentage | `AVG(CASE...)*100` | boolean mean ×100 | `avg(when(...))*100` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Queries

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: queries_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: queries_spark